In [3]:
# ALL BASED ON PROOF OF 7.1 BENNETT ET AL !!!!!!!!!! USED FOR t(p^2)
import pandas as pd

# function to obtain potential d in t(p^d-1)=l
def get_d(l):
    eqn = l * ((l ** 2)-1) # theorem 1.1 by balakrishnan gives this eqn, d and odd prime divisors of this
    d = [i for i in eqn.prime_divisors() if (i % 2 != 0 and i != l)]
    return d

def congruence_checking(l,d_list): # lemma A.1 Goto, Ramanujan's prime power congruences
    # remaining d after congruence checking
    d_left = []
    for d in d_list:
        first,second,third,fourth = False,False,False,False
         # first congruence
        if (l % 12 in (0,1,d % 12)):
            first = True
            
        # second congruence
        if (d % 4) == 1:
            if (l % 5 in (0,1,d % 5)):
                second = True
        if (d % 4) == 3:
            if (l % 5 in (0,1,2,3,d % 5)):
                second = True

        # third congruence
        if (d % 6) == 1:
            if (l % 7 in (0,1,d % 7)):
                third = True
        if (d % 6) in (3,5):
            if (l % 7 in (0,1,2,4,d,(2*d) % 7,(4*d) % 7)):
                third = True
                
        # fourth congruence
        if (d % 6) == 1:
            if (l % 23 in (1,d % 23)):
                fourth = True
        if (d % 6) == 3:
            if (l % 23 in (0,1,d % 23)):
                fourth = True
        if (d % 6) == 5:
            if (l % 23 in (1,-1,d % 23)):
                fourth = True

        if all([first,second,third,fourth]): # check if all congruences are true for this potential d
            d_left.append(d)
    return d_left

primes_list = list(prime_range(252,1000))
neg_and_pos_primes = list()
for l in primes_list:
    neg_and_pos_primes.append(l)
    neg_and_pos_primes.append(-l)

data = dict()
for l in neg_and_pos_primes:
    d_list = get_d(l)
    
    filtered_d = congruence_checking(l, d_list)
    data[l] = list(filtered_d)
data

{257: [],
 -257: [],
 263: [],
 -263: [],
 269: [],
 -269: [],
 271: [],
 -271: [],
 277: [3, 23],
 -277: [],
 281: [5],
 -281: [],
 283: [],
 -283: [],
 293: [],
 -293: [],
 307: [],
 -307: [],
 311: [],
 -311: [],
 313: [],
 -313: [],
 317: [],
 -317: [],
 331: [],
 -331: [],
 337: [],
 -337: [],
 347: [],
 -347: [],
 349: [],
 -349: [],
 353: [],
 -353: [],
 359: [],
 -359: [],
 367: [],
 -367: [],
 373: [],
 -373: [],
 379: [],
 -379: [],
 383: [],
 -383: [],
 389: [],
 -389: [],
 397: [],
 -397: [],
 401: [],
 -401: [],
 409: [],
 -409: [],
 419: [],
 -419: [],
 421: [7],
 -421: [],
 431: [],
 -431: [],
 433: [],
 -433: [],
 439: [],
 -439: [],
 443: [],
 -443: [17],
 449: [],
 -449: [],
 457: [],
 -457: [],
 461: [5],
 -461: [],
 463: [],
 -463: [],
 467: [],
 -467: [],
 479: [],
 -479: [],
 487: [],
 -487: [],
 491: [],
 -491: [],
 499: [],
 -499: [],
 503: [],
 -503: [3],
 509: [],
 -509: [],
 521: [],
 -521: [],
 523: [],
 -523: [],
 541: [],
 -541: [],
 547: [],
 -547: [],
 5

In [3]:
# ALL BASED ON PROOF OF 7.1 BENNETT ET AL !!!!!!!!!!

# get coefficient of fundamental unit

K.<sqrt277> = QuadraticField(877)

U = K.unit_group()
fundamental_unit = U.fundamental_units()[0]
print(fundamental_unit)
# Factor a specific integer/rational within the number field, CHECK THAT 23 SPLITS IN RING OF INTEGERS FOR 277,601,877 

q = K.primes_above(23)[0]

for a in range(0, 11):
    F = q.residue_field()          # Create the residue field (GF(23))
    phi = F.reduction_map()        # Get the reduction homomorphism K -> GF(23)
    
    val_reduced = 2+phi(sqrt277)    # Reduce sqrt(277) mod q
    unit_reduced = phi(fundamental_unit^a)
    if val_reduced == unit_reduced:
        print("a must be equal to: ", a)


-8149*sqrt277 + 241326
a must be equal to:  4


In [4]:
# FUNDAMENTAL UNIT CHECKER
x = polygen(QQ)
K.<w> = NumberField(x^2 - 601)
U = K.unit_group()
fundamental_unit = U.fundamental_units()[0]
print(fundamental_unit)

5689030769845*w - 139468303679532


In [7]:
# solve thue equations t(p^2) = l,
def generate_and_solve_thue(d, a, N=11, target=1):
    x = polygen(QQ)
    K.<w> = NumberField(x^2 - d)
    U = K.unit_group()
    fundamental_unit = U.fundamental_units()[0]
    unit_power = (fundamental_unit)^a
    
    c0 = QQ(unit_power[0])  # Rational coefficient
    c1 = QQ(unit_power[1])  # Coefficient of sqrt(d)
    print(c0, c1)
    
    """
    Constructs the Thue equation arising from the coefficient of sqrt(d) in
    (c0 + c1*sqrt(d)) * (U + V*sqrt(d))^N = Target, and solves it.
    
    Parameters:
    - d (int): Square-free integer for Q(sqrt(d)).
    - c0, c1 (QQ): Coefficients of the multiplier unit (c0 + c1*sqrt(d)).
    - N (int): Exponent / degree of the binary form.
    - target (int): Right-hand side integer of the Thue equation (default 1).
    
    Returns:
    - F: The bivariate polynomial F(U, V) representing the coefficient of sqrt(d).
    - solutions: List of integer pairs [U, V] satisfying F(U, V) = target.
    """
    # 1. Setup the polynomial ring for U, V over Rational Field
    R.<U, V> = PolynomialRing(QQ)
    
    # 2. Compute (U + V*sqrt(d))^N algebraically using Binomial Theorem
    #    (U + V*sqrt(d))^N = A(U,V) + B(U,V)*sqrt(d)
    A = R(0)
    B = R(0)
    
    for k in range(N + 1):
        term = binomial(N, k) * (U^(N - k)) * (V^k)
        if k % 2 == 0: # Even powers of sqrt(d) give rational integers d^(k/2)
            A += term * (d^(k // 2))
        else:          # Odd powers of sqrt(d) give d^((k-1)/2) * sqrt(d)
            B += term * (d^((k - 1) // 2))
            
    # 3. Multiply by (c0 + c1*sqrt(d)):
    #    (c0 + c1*sqrt(d)) * (A + B*sqrt(d)) = (c0*A + c1*B*d) + (c0*B + c1*A)*sqrt(d)
    #    We extract the coefficient of sqrt(d):
    F = c0 * B + c1 * A
    #reduce polynomial
    F_min = F.reduced_form()[0]
    print(F.reduced_form()[1:])

    # 4. Convert F(U, V) to univariate form f(x) = F(x, 1) for the PARI/GP solver
    f_univariate = F_min.subs({V: 1})
    
    # 5. Solve using PARI's thue solver
    pari_sols = gp.thue(f_univariate, target)
    solutions = pari_sols.sage()
    return F_min, solutions

print(generate_and_solve_thue(277,0))


1 0
([1 0]
[0 1],)
(11*U^10*V + 45705*U^8*V^3 + 35448798*U^6*V^5 + 7013797890*U^4*V^7 + 323803669255*U^2*V^9 + 1630793025157*V^11, [])


In [8]:
k.<sqrt277> = QuadraticField(-503)
print(k.disc())
print(kronecker(k.disc(),23))

-503
1


In [7]:
def generate_and_solve_thue(d, a, N=11, target=1):
    x = polygen(QQ)
    K.<w> = NumberField(x^2 - d)
    U_grp = K.unit_group()
    fundamental_unit = U_grp.fundamental_units()[0]
    unit_power = K(fundamental_unit)^a
    
    # Extract coefficients in the {1, sqrt(d)} basis safely (even when d % 4 == 1)
    coeffs = unit_power.list()
    c0 = QQ(coeffs[0])
    c1 = QQ(coeffs[1]) if len(coeffs) > 1 else QQ(0)
    
    # 1. Setup the polynomial ring for U, V over Rational Field
    R.<U, V> = PolynomialRing(QQ)
    
    # 2. Compute (U + V*sqrt(d))^N = A(U,V) + B(U,V)*sqrt(d)
    A = R(0)
    B = R(0)
    for k in range(N + 1):
        term = binomial(N, k) * (U^(N - k)) * (V^k)
        if k % 2 == 0:
            A += term * (d^(k // 2))
        else:
            B += term * (d^((k - 1) // 2))
            
    # 3. Extract the coefficient of sqrt(d)
    F = c0 * B + c1 * A
    
    # Reduce polynomial and capture the transformation matrix M
    F_min, M = F.reduced_form()

    # 4. Convert F_min(U', V') to univariate form for PARI/GP
    f_univariate = F_min.subs({V: 1})
    
    # 5. Solve using PARI's thue solver
    pari_sols = gp.thue(f_univariate, target)
    reduced_solutions = pari_sols.sage()
    
    # 6. Recover original (U, V) = M * [U', V']^T and verify
    original_solutions = []
    for sol in reduced_solutions:
        uv_orig = M * vector(ZZ, sol)
        u_orig, v_orig = uv_orig[0], uv_orig[1]
        
        # Sanity check: verify F(u_orig, v_orig) == target
        assert F(u_orig, v_orig) == target, f"Mismatch for {sol}"
        original_solutions.append([u_orig, v_orig])
        
    return F, original_solutions

generate_and_solve_thue(877,4)

(-916234515879791945688*U^11 + 298468930984434292759787*U^10*V - 44194571873461764500260680*U^9*V^2 + 3926358787100233121254997985*U^8*V^3 - 232551837198155804800371698160*U^7*V^4 + 9641566637603332452553773051966*U^6*V^5 - 285527145711895697133896371000848*U^5*V^6 + 6039752815127230400635470690410130*U^4*V^7 - 89431180996190187995152541917051320*U^3*V^8 + 882810536477763510226217965914947335*U^2*V^9 - 5228743048910586324783251950750267176*U*V^10 + 14076815281654519972152602838316523869*V^11,
 [])